# Geocodificação híbrida barata

Este notebook aplica uma estratégia híbrida e barata de geocodificação, com cache local e classificação em `alta`, `media` ou `baixa` confiança.

## Saídas principais

- `estabelecimentos_geocodificados.parquet`
- `qualidade_geocodificacao.csv`
- `geocodificacao_cache.parquet`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

try:
    import pyarrow  # noqa: F401
except ImportError:
    !pip -q install pyarrow
    import pyarrow  # noqa: F401

try:
    from geopy.extra.rate_limiter import RateLimiter
    from geopy.geocoders import Nominatim
except ImportError:
    !pip -q install geopy
    from geopy.extra.rate_limiter import RateLimiter
    from geopy.geocoders import Nominatim

SNAPSHOT_MES = '2026-04'
CIDADE_ALVO = 'Praia Grande'
UF_ALVO = 'SP'
PAIS_ALVO = 'Brasil'
SNAPSHOT = Path('/content/dados_brutos/cnpj') / SNAPSHOT_MES
PASTA_PREPARADO = SNAPSHOT / 'processado' / 'preparado'
ESTAB_PREPARADOS = PASTA_PREPARADO / 'estabelecimentos_preparados.parquet'
CACHE_GEOCODIFICACAO = PASTA_PREPARADO / 'geocodificacao_cache.parquet'
SAIDA_GEOCODIFICADA = PASTA_PREPARADO / 'estabelecimentos_geocodificados.parquet'
SAIDA_QUALIDADE = PASTA_PREPARADO / 'qualidade_geocodificacao.csv'

if not ESTAB_PREPARADOS.exists():
    raise FileNotFoundError(f'Arquivo preparado ausente: {ESTAB_PREPARADOS}')

PAUSA_NOMINATIM_SEGUNDOS = 1.1
ESTAB_PREPARADOS

In [ ]:
estabelecimentos = pd.read_parquet(ESTAB_PREPARADOS)
estabelecimentos.shape

In [ ]:
def limpar_texto(valor: object) -> str:
    if valor is None:
        return ''
    return ' '.join(str(valor).strip().split())

def consulta_endereco_completo(linha: pd.Series) -> str:
    partes = [
        limpar_texto(linha.get('logradouro_limpo')),
        limpar_texto(linha.get('numero_limpo')),
        limpar_texto(linha.get('bairro_limpo')),
        CIDADE_ALVO,
        UF_ALVO,
        PAIS_ALVO,
    ]
    partes = [parte for parte in partes if parte]
    return ', '.join(partes)

def consulta_por_cep(linha: pd.Series) -> str:
    cep = limpar_texto(linha.get('cep_limpo'))
    if len(cep) != 8:
        return ''
    return ', '.join([cep, CIDADE_ALVO, UF_ALVO, PAIS_ALVO])

def consulta_logradouro_bairro(linha: pd.Series) -> str:
    partes = [limpar_texto(linha.get('logradouro_limpo')), limpar_texto(linha.get('bairro_limpo')), CIDADE_ALVO, UF_ALVO, PAIS_ALVO]
    partes = [parte for parte in partes if parte]
    return ', '.join(partes)

def consulta_bairro(linha: pd.Series) -> str:
    bairro = limpar_texto(linha.get('bairro_limpo'))
    if not bairro:
        return ''
    return ', '.join([bairro, CIDADE_ALVO, UF_ALVO, PAIS_ALVO])


In [ ]:
enderecos = estabelecimentos[['chave_endereco', 'logradouro_limpo', 'numero_limpo', 'bairro_limpo', 'cep_limpo']].drop_duplicates().copy()
enderecos['consulta_endereco_completo'] = enderecos.apply(consulta_endereco_completo, axis=1)
enderecos['consulta_cep'] = enderecos.apply(consulta_por_cep, axis=1)
enderecos['consulta_logradouro_bairro'] = enderecos.apply(consulta_logradouro_bairro, axis=1)
enderecos['consulta_bairro'] = enderecos.apply(consulta_bairro, axis=1)

consultas = []
for _, linha in enderecos.iterrows():
    candidatas = [
        ('endereco_completo', 'alta', 1, linha['consulta_endereco_completo']),
        ('cep', 'media', 2, linha['consulta_cep']),
        ('logradouro_bairro', 'media', 3, linha['consulta_logradouro_bairro']),
        ('bairro', 'baixa', 4, linha['consulta_bairro']),
    ]
    for metodo, confianca, ordem, consulta in candidatas:
        if consulta:
            consultas.append({
                'chave_endereco': linha['chave_endereco'],
                'metodo_planejado': metodo,
                'confianca_planejada': confianca,
                'ordem': ordem,
                'consulta_geocodificacao': consulta,
            })

consultas = pd.DataFrame(consultas).drop_duplicates()
consultas.shape

In [ ]:
colunas_cache = ['consulta_geocodificacao', 'metodo_planejado', 'latitude', 'longitude', 'endereco_encontrado', 'sucesso_consulta', 'timestamp_consulta']
if CACHE_GEOCODIFICACAO.exists():
    cache = pd.read_parquet(CACHE_GEOCODIFICACAO)
else:
    cache = pd.DataFrame(columns=colunas_cache)

consultas_unicas = consultas[['consulta_geocodificacao', 'metodo_planejado']].drop_duplicates()
pendentes = consultas_unicas.merge(
    cache[['consulta_geocodificacao', 'metodo_planejado']].drop_duplicates(),
    on=['consulta_geocodificacao', 'metodo_planejado'],
    how='left',
    indicator=True,
)
pendentes = pendentes.loc[pendentes['_merge'] == 'left_only', ['consulta_geocodificacao', 'metodo_planejado']]

len(pendentes)

In [ ]:
geolocator = Nominatim(user_agent='geobusiness-poc')
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=PAUSA_NOMINATIM_SEGUNDOS, swallow_exceptions=True)

novos_registros = []
for indice, linha in pendentes.reset_index(drop=True).iterrows():
    consulta = linha['consulta_geocodificacao']
    metodo = linha['metodo_planejado']
    local = geocode(consulta, country_codes='br', addressdetails=True, exactly_one=True, timeout=15)

    novos_registros.append({
        'consulta_geocodificacao': consulta,
        'metodo_planejado': metodo,
        'latitude': np.nan if local is None else float(local.latitude),
        'longitude': np.nan if local is None else float(local.longitude),
        'endereco_encontrado': '' if local is None else str(local.address),
        'sucesso_consulta': bool(local is not None),
        'timestamp_consulta': pd.Timestamp.utcnow(),
    })

    if (indice + 1) % 25 == 0:
        print(f'Consultas processadas: {indice + 1} / {len(pendentes)}')

if novos_registros:
    cache = pd.concat([cache, pd.DataFrame(novos_registros)], ignore_index=True)
    cache = cache.drop_duplicates(subset=['consulta_geocodificacao', 'metodo_planejado'], keep='last')
    cache.to_parquet(CACHE_GEOCODIFICACAO, index=False)

cache.shape

In [ ]:
resolucao = consultas.merge(cache, on=['consulta_geocodificacao', 'metodo_planejado'], how='left')
resolucao = resolucao.sort_values(['chave_endereco', 'ordem'])
resolucao_sucesso = resolucao.loc[resolucao['sucesso_consulta'] == True].drop_duplicates('chave_endereco')

estabelecimentos_geocodificados = estabelecimentos.merge(
    resolucao_sucesso[['chave_endereco', 'metodo_planejado', 'confianca_planejada', 'latitude', 'longitude', 'endereco_encontrado']],
    on='chave_endereco',
    how='left',
)
estabelecimentos_geocodificados['metodo_geocodificacao'] = estabelecimentos_geocodificados['metodo_planejado'].fillna('nao_localizado')
estabelecimentos_geocodificados['confianca_geografica'] = estabelecimentos_geocodificados['confianca_planejada'].fillna('baixa')
estabelecimentos_geocodificados['precisa_revisao_geocodificacao'] = estabelecimentos_geocodificados['confianca_geografica'] != 'alta'
estabelecimentos_geocodificados[['metodo_geocodificacao', 'confianca_geografica', 'latitude', 'longitude']].head()

In [ ]:
estabelecimentos_geocodificados.to_parquet(SAIDA_GEOCODIFICADA, index=False)
qualidade = estabelecimentos_geocodificados.groupby(['confianca_geografica', 'metodo_geocodificacao'], dropna=False).size().reset_index(name='quantidade')
qualidade['percentual'] = qualidade['quantidade'] / max(len(estabelecimentos_geocodificados), 1)
qualidade.to_csv(SAIDA_QUALIDADE, index=False)

qualidade